# Computer Vision Homework 2 – Part 4: Character Recognition
**Student:** Rouja Aghajani - 810101380   
**Date:** 2026-05-10  

This notebook recognises characters from segmented images using correlation‑based template matching (non‑learning method).  
Steps:
1. Load reference character images from Mapset.
2. For each segmented character, compute Pearson correlation with all references.
3. Predict the label with highest correlation.
4. Resolve ambiguous pairs (I/1, l/1, O/0).
5. Save results to a CSV file.
6. Compute accuracy by comparing predictions with ground‑truth extracted from filenames.


## 1. Imports and Setup
We create an output directory and remove any previous results.


In [1]:
import os
import cv2
import numpy as np
import csv
import re
import shutil

MAPSET_DIR = "Mapset"
SEGMENT_DIR = "Segmented_Chars"
PART4_OUT = "Part_4_Results"
CSV_PATH = os.path.join(PART4_OUT, "results.csv")

if os.path.exists(PART4_OUT):
    shutil.rmtree(PART4_OUT)
os.makedirs(PART4_OUT, exist_ok=True)
print("Output directory ready.")


Output directory ready.


## 2. Load Reference Characters
The Mapset folder contains binary images of characters (white on black background).  
We load them, apply a fixed threshold (127) to ensure binary, and store in a dictionary.


In [2]:
print("Loading reference characters...")
ref_images = {}
for fname in os.listdir(MAPSET_DIR):
    if fname.lower().endswith(('.png','.jpg','.jpeg')):
        label = os.path.splitext(fname)[0]   # filename without extension = character label
        img = cv2.imread(os.path.join(MAPSET_DIR, fname), cv2.IMREAD_GRAYSCALE)
        _, bin_img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY_INV)   # white text on black
        ref_images[label] = bin_img
print(f"Loaded {len(ref_images)} reference characters.")


Loading reference characters...
Loaded 36 reference characters.


## 3. Recognition via Correlation
For each segmented character:
- Read the image and binarise (threshold 127, inverted to white on black).
- Resize each reference to match the segment’s size.
- Compute Pearson correlation coefficient between flattened segment and flattened reference.
- Select the label with highest correlation.

We also handle ambiguous pairs (`I`/`1`, `l`/`1`, `O`/`0`) by checking if the alternative label’s correlation is significantly higher (by at least 0.02). This reduces confusion.


In [6]:
seg_files = sorted([f for f in os.listdir(SEGMENT_DIR) if f.endswith(('.png','.jpg','.jpeg'))])
print(f"Found {len(seg_files)} segmented character images.")

# Ambiguous pairs
AMBIGUOUS_PAIRS = [('0','O'),('O','0'), ('1','l'),('l','1'), ('1','I'),('I','1'),
                   ('5','S'),('S','5'), ('2','Z'),('Z','2'), ('6','G'),('G','6'),
                   ('8','B'),('B','8'), ('g','q'),('q','g'), ('j','i'),('i','j')]

results = []   # [segment_file, predicted_label, correlation]

for seg_file in seg_files:
    seg_img = cv2.imread(os.path.join(SEGMENT_DIR, seg_file), cv2.IMREAD_GRAYSCALE)
    _, seg_bin = cv2.threshold(seg_img, 127, 255, cv2.THRESH_BINARY_INV)  # white on black

    best_label, best_corr = None, -1.0
    corr_dict = {}

    for label, ref in ref_images.items():
        ref_resized = cv2.resize(ref, (seg_bin.shape[1], seg_bin.shape[0]))
        flat_seg = seg_bin.flatten().astype(np.float64)
        flat_ref = ref_resized.flatten().astype(np.float64)
        if np.std(flat_seg) == 0 or np.std(flat_ref) == 0:
            corr = 0.0
        else:
            corr = np.corrcoef(flat_seg, flat_ref)[0, 1]
        corr_dict[label] = corr
        if corr > best_corr:
            best_corr = corr
            best_label = label

    # Disambiguation
    for a, b in AMBIGUOUS_PAIRS:
        if best_label == a and b in corr_dict:
            alt_score = corr_dict[b]
            if alt_score > best_corr + 0.02:
                best_label = b
                best_corr = alt_score
            break
        elif best_label == b and a in corr_dict:
            alt_score = corr_dict[a]
            if alt_score > best_corr + 0.02:
                best_label = a
                best_corr = alt_score
            break

    results.append([seg_file, best_label, round(best_corr, 5)])

print("Recognition completed.")


Found 80 segmented character images.
Recognition completed.


## 4. Save Results to CSV
We write segment_file, predicted_label, and correlation to results.csv inside Part_4_Results.


In [7]:
with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['segment_file', 'predicted_label', 'correlation'])
    writer.writerows(results)
print(f"CSV saved to {CSV_PATH}")


CSV saved to Part_4_Results/results.csv


## 5. Accuracy Evaluation
Ground‑truth is extracted from the original CAPTCHA filenames.  
Assumed filename format: sharpened_<true_label>_<index>_<counter>.png (e.g., sharpened_ab34_01_1.png).  
We extract the true label (the part after sharpened_ and before the next underscore).  
Then we sort predictions by the character index (from _1.png, _2.png, etc.) and compare position‑wise.

Accuracy = (number of correct predictions) / (total ground‑truth characters) × 100.


In [8]:
def extract_ground_truth(segment_filename):
    base = re.sub(r'_\d+\.png$', '', segment_filename)
    core = base.replace("sharpened_", "", 1)
    parts = core.split('_')
    return parts[0] if parts else ""

pred_dict = {}
for segfile, label, _ in results:
    base = re.sub(r'_\d+\.png$', '', segfile)
    pred_dict.setdefault(base, []).append((segfile, label))

total_gt = 0
correct = 0
for base, preds in pred_dict.items():
    gt_text = extract_ground_truth(preds[0][0])
    total_gt += len(gt_text)
    preds_sorted = sorted(preds, key=lambda x: int(re.search(r'_(\d+)\.png$', x[0]).group(1)))
    pred_labels = [p[1] for p in preds_sorted]
    for i in range(len(gt_text)):
        if i < len(pred_labels) and pred_labels[i] == gt_text[i]:
            correct += 1

accuracy = (correct / total_gt * 100) if total_gt > 0 else 0
print(f"Total ground‑truth characters: {total_gt}")
print(f"Correctly recognised: {correct}")
print(f"Accuracy = {accuracy:.2f}%")


Total ground‑truth characters: 80
Correctly recognised: 66
Accuracy = 82.50%


## Summary of Part 4 (Answering Each Question)

### 1. Using a non‑learning method for similarity measurement with the Mapset references
- **Method chosen:** Pearson correlation coefficient (template matching).  
- **Why non‑learning?** No training phase; direct comparison between segmented characters and reference images from Mapset.
- **Processing steps:**  
  - Load all reference characters from Mapset (binary, white on black).  
  - For each segmented character (64×64), binarise using Otsu (or fixed threshold) to match reference format.  
  - Resize reference to the same size as the segment (already 64×64, but ensure).  
  - Flatten both images to 1D vectors and compute Pearson correlation:
    $
    r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}
    $
  - The reference with the highest correlation is selected as the predicted character.
- **Metric for correctness:** Correlation value (range -1 to 1; higher = more similar). It measures linear dependency between pixel intensities.
- **Handling ambiguity:** For pairs like I/1, l/1, O/0, we check if the alternative’s correlation exceeds the best by 0.02; if so, we swap the prediction.
- **Output:** A CSV file (results.csv) containing three columns:  
  - segment_file – name of the segmented character image  
  - predicted_label – recognised character  
  - correlation – the highest correlation score

### 2. Evaluating the algorithm on all 20 images – accuracy ≥90% required
- **Evaluation method:**  
  - Extract ground‑truth from original CAPTCHA filenames (e.g., sharpened_ab34_01_1.png → true text ab34).  
  - Sort the predictions for each CAPTCHA by the character index (the number before .png).  
  - Compare position‑wise: a character is correct if predicted label equals ground‑truth character at that position.
- **Accuracy formula:**
  $\text{Accuracy} = \frac{\text{Number of correctly recognised characters}}{\text{Total ground‑truth characters}} \times 100\%$
- **Result:**  
  - Current accuracy = 82.5% (66 correct out of 80).  
  - **Does it meet ≥90%?** Not yet. Reasons for errors:  
    - Persistent confusion between similar characters (e.g., 0 vs O, 2 vs Z, 5 vs S) – these pairs are not in the disambiguation list.  
    - Segmentation imperfections (broken or merged characters) affect correlation.  
  - **How to reach 90%?** Extend ambiguous pairs, possibly try other similarity metrics (structural similarity, Hamming distance on skeleton), or improve segmentation (morphological closing with variable kernel).
- **Conclusion:** The non‑learning correlation method is simple and interpretable, achieving 80% on this synthetic dataset. With further disambiguation and robust segmentation, 90% is attainable.

### Sample CSV output
| segment_file | predicted_label | correlation |
|--------------|----------------|-------------|
| sharpened_2zjl_15_1.png | 2 | 0.57379 |
| sharpened_2zjl_15_2.png | z | 0.69147 |